# Intialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import DateType


# Read Bronze Table

In [0]:
df = spark.table("workspace.bronze.prd_info")


# Exploring Data

In [0]:
display(df)

print("📊 تقرير القيم المفقودة:")
display(df.pandas_api().isnull().sum(axis=0))

# Silver Transformations

## Trimming


In [0]:
trim_exprs = [
    F.trim(F.col(c)).alias(c) if t == 'string' else F.col(c)
    for c, t in df.dtypes
]
# تنفيذ الخطة بخبطة واحدة
df = df.select(*trim_exprs)



## Product Key Parsing & Cost Cleanup


In [0]:
df = (
    df
    # أ. فصل كود القسم (أول 5 حروف) واستبدال الشرطة بـ Underscore
    .withColumn("cat_id", F.regexp_replace(F.substring(F.col("prd_key"), 1, 5), "-", "_"))
    
    # ب. الاحتفاظ بباقي كود المنتج (من الحرف الـ 7 للآخر)
    .withColumn("prd_key", F.substring(F.col("prd_key"), 7, F.length(F.col("prd_key")))).withColumn("prd_start_dt", F.col("prd_start_dt").cast(DateType()))
)

## Product Line Normalization

In [0]:
df=df.withColumn("prd_line",
        F.when(F.upper(F.col("prd_line")) == "M", "Mountain")
         .when(F.upper(F.col("prd_line")) == "R", "Road")
         .when(F.upper(F.col("prd_line")) == "S", "Other Sales")
         .when(F.upper(F.col("prd_line")) == "T", "Touring")
         .otherwise("n/a")
    )

## Rename Columns


In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Date Casting

In [0]:
df = df.withColumn("start_date", F.col("start_date").cast(DateType()))

## Sanity checks of dataframe

In [0]:
df.limit(10).display()

print("📊 تقرير القيم المفقودة:")
display(df.pandas_api().isnull().sum(axis=0))

# Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")

##  Sanity checks of silver table

In [0]:
%sql
select * from workspace.silver.crm_products
limit 10;